In [ ]:
import os
import gc

import numpy as np
import pandas as pd

import cfospy

In [ ]:
class mapping_to_atlas():
    def __init__(self, data_atlas_path, ants_dir_name, data_ants_path, before_ants_file):
        self.ants_voxel_unit = Resize_um
        self.original_voxel_unit = Resize_um
        self.img_voxel_unit = Resize_um

        self.atlas_tif_path = data_atlas_path
        self.atlas_nii_path = self.atlas_tif_path.replace(".tif", ".nii.gz")

        # Output / working directories and paths
        self.ants_dst_dir = os.path.join(data_ants_path, ants_dir_name)
        self.sample_resize_img_path = os.path.join(data_ants_path, before_ants_file)

        # From .tif to .nii path mapping for the sample
        self.sample_nii_path = os.path.join(
            data_ants_path, before_ants_file.replace(".tif", ".nii.gz")
        )

        self.before_ants_np_path = os.path.join(data_ants_path, "all_points_um.npy")

        if not os.path.exists(self.ants_dst_dir):
            print(f"make ants folder {self.ants_dst_dir}")
            os.makedirs(self.ants_dst_dir)
        else:
            print(f"{self.ants_dst_dir} already exists")

        self.moving_nii_path = self.sample_nii_path
        self.output_nii_path = os.path.join(self.ants_dst_dir, "after_ants.nii.gz")

        # ANTs binary prefix
        self.prefix_ants = "/opt/ANTs/bin/"

In [ ]:
src = "/path/to/source_dir"
dst = "/path/to/output_dir"
cfos_dir = os.path.join(src, "circadian_1st", "circadian_1st_Reconst")
savedir = os.path.join(dst, "cfos_app")

In [ ]:
# Collect unique sample names

CT_li = np.arange(0, 48, 4)  # circadian time points (CT0–44, every 4 h)
sample_ids = np.arange(1, 7, 1)

reconsts = os.listdir(cfos_dir)
sample_names = []

for CT in CT_li:
    for sample_id in sample_ids:
        sample = f"CT{CT}_{str(sample_id).zfill(2)}"
        for reconst in reconsts:
            if sample in reconst:
                sample_names.append(sample)

print(len(sample_names))

In [ ]:
# Load atlas data
rdir = os.path.join(src, "CUBIC_R_atlas_ver5")
vx = 50
ca = cfospy.analysis.read_atlas_data(rdir, vx)
print(f"{len(ca.ID_all)} regions")

In [ ]:
# Annotate and summarize AI-predicted cells by experiment

deconv_fs = ["circadian_1st/circadian_1st_Deconv", "circadian_2nd/circadian_2nd_Deconv"]
exps = ["1st", "2nd"]
ants_dir_name = "ANTsR50"

fpr = 0.5
postfix = f"output_peak_ratioI_fpr{fpr}"
moving_points_paths = [os.path.join(dst, f"{i}_{postfix}") for i in deconv_fs]
moved_points_id_csv = f"coordinates_ai_id_fpr{fpr}.csv"

for l, exp in enumerate(exps):
    cell_nums = np.zeros(len(sample_names), dtype="uint64")
    cell_intenses = np.zeros(len(sample_names), dtype="uint64")

    moving_points_path = moving_points_paths[l]
    moving_points_csv_li = os.listdir(moving_points_path)

    for i, sample in enumerate(sample_names):
        for f in moving_points_csv_li:
            if sample in f.replace("cfos_", ""):
                break
        print(f)

        # Load transformed cell table after ANTs registration
        moved_points_dir = os.path.join(savedir, exp, sample, "SYTOX-G", ants_dir_name)
        moved_points_file = os.path.join(moved_points_dir, moved_points_id_csv)

        post_cell = pd.read_csv(moved_points_file)
        post_cell = post_cell[post_cell["predict"] == 1]
        post_cell = post_cell[post_cell["ID"] != 0]

        cell_nums[i] = len(post_cell)
        cell_intenses[i] = np.sum(post_cell["intensity"])

    np.save(os.path.join(savedir, exp, f"total_cell_nums_fpr{fpr}"), cell_nums)
    np.save(os.path.join(savedir, exp, f"total_cell_intenses_fpr{fpr}"), cell_intenses)

# Combine results from 1st and 2nd experiments
t_cells_1 = np.load(os.path.join(savedir, f"1st/total_cell_nums_fpr{fpr}.npy"))
t_cells_2 = np.load(os.path.join(savedir, f"2nd/total_cell_nums_fpr{fpr}.npy"))

t_cell_T = np.append(t_cells_1, t_cells_2)
print(t_cell_T)

np.save(os.path.join(savedir, f"total_cell_nums_fpr{fpr}"), t_cell_T)

cfos_CT12_06
/home/gpu_data/data8/cfos_app/2nd/CT12_06/SYTOX-G/ANTsR50/ already exists
cfos_CT16_01
/home/gpu_data/data8/cfos_app/2nd/CT16_01/SYTOX-G/ANTsR50/ already exists
cfos_CT16_02
/home/gpu_data/data8/cfos_app/2nd/CT16_02/SYTOX-G/ANTsR50/ already exists
cfos_CT16_03
/home/gpu_data/data8/cfos_app/2nd/CT16_03/SYTOX-G/ANTsR50/ already exists
cfos_CT16_04
/home/gpu_data/data8/cfos_app/2nd/CT16_04/SYTOX-G/ANTsR50/ already exists
cfos_CT16_05
/home/gpu_data/data8/cfos_app/2nd/CT16_05/SYTOX-G/ANTsR50/ already exists
cfos_CT16_06
/home/gpu_data/data8/cfos_app/2nd/CT16_06/SYTOX-G/ANTsR50/ already exists
cfos_CT20_01
/home/gpu_data/data8/cfos_app/2nd/CT20_01/SYTOX-G/ANTsR50/ already exists
cfos_CT20_02
/home/gpu_data/data8/cfos_app/2nd/CT20_02/SYTOX-G/ANTsR50/ already exists
cfos_CT20_03
/home/gpu_data/data8/cfos_app/2nd/CT20_03/SYTOX-G/ANTsR50/ already exists
cfos_CT20_04
/home/gpu_data/data8/cfos_app/2nd/CT20_04/SYTOX-G/ANTsR50/ already exists
cfos_CT20_05
/home/gpu_data/data8/cfos_app/

cfos_CT28_05
/home/gpu_data/data8/cfos_app/1st/CT28_05/SYTOX-G/ANTsR50/ already exists
cfos_CT28_06
/home/gpu_data/data8/cfos_app/1st/CT28_06/SYTOX-G/ANTsR50/ already exists
cfos_CT32_01
/home/gpu_data/data8/cfos_app/1st/CT32_01/SYTOX-G/ANTsR50/ already exists
cfos_CT32_02
/home/gpu_data/data8/cfos_app/1st/CT32_02/SYTOX-G/ANTsR50/ already exists
cfos_CT32_03
/home/gpu_data/data8/cfos_app/1st/CT32_03/SYTOX-G/ANTsR50/ already exists
cfos_CT32_04
/home/gpu_data/data8/cfos_app/1st/CT32_04/SYTOX-G/ANTsR50/ already exists
cfos_CT32_05
/home/gpu_data/data8/cfos_app/1st/CT32_05/SYTOX-G/ANTsR50/ already exists
cfos_CT32_06
/home/gpu_data/data8/cfos_app/1st/CT32_06/SYTOX-G/ANTsR50/ already exists
cfos_CT36_01
/home/gpu_data/data8/cfos_app/1st/CT36_01/SYTOX-G/ANTsR50/ already exists
cfos_CT36_02
/home/gpu_data/data8/cfos_app/1st/CT36_02/SYTOX-G/ANTsR50/ already exists
cfos_CT36_03
/home/gpu_data/data8/cfos_app/1st/CT36_03/SYTOX-G/ANTsR50/ already exists
cfos_CT36_04
/home/gpu_data/data8/cfos_app/

cfos_CT44_04
/home/gpu_data/data8/cfos_app/2nd/CT44_04/SYTOX-G/ANTsR50/ already exists
cfos_CT44_05
/home/gpu_data/data8/cfos_app/2nd/CT44_05/SYTOX-G/ANTsR50/ already exists
cfos_CT44_06
/home/gpu_data/data8/cfos_app/2nd/CT44_06/SYTOX-G/ANTsR50/ already exists
cfos_CT0_01
/home/gpu_data/data8/cfos_app/1st/CT0_01/SYTOX-G/ANTsR50/ already exists
cfos_CT0_02
/home/gpu_data/data8/cfos_app/1st/CT0_02/SYTOX-G/ANTsR50/ already exists
cfos_CT0_03
/home/gpu_data/data8/cfos_app/1st/CT0_03/SYTOX-G/ANTsR50/ already exists
cfos_CT0_04
/home/gpu_data/data8/cfos_app/1st/CT0_04/SYTOX-G/ANTsR50/ already exists
cfos_CT0_05
/home/gpu_data/data8/cfos_app/1st/CT0_05/SYTOX-G/ANTsR50/ already exists
cfos_CT0_06
/home/gpu_data/data8/cfos_app/1st/CT0_06/SYTOX-G/ANTsR50/ already exists
cfos_CT4_01
/home/gpu_data/data8/cfos_app/1st/CT4_01/SYTOX-G/ANTsR50/ already exists
cfos_CT4_02
/home/gpu_data/data8/cfos_app/1st/CT4_02/SYTOX-G/ANTsR50/ already exists
cfos_CT4_03
/home/gpu_data/data8/cfos_app/1st/CT4_03/SYTOX-

cfos_CT12_03
/home/gpu_data/data8/cfos_app/2nd/CT12_03/SYTOX-G/ANTsR50/ already exists
cfos_CT12_04
/home/gpu_data/data8/cfos_app/2nd/CT12_04/SYTOX-G/ANTsR50/ already exists
cfos_CT12_05
/home/gpu_data/data8/cfos_app/2nd/CT12_05/SYTOX-G/ANTsR50/ already exists
cfos_CT12_06
/home/gpu_data/data8/cfos_app/2nd/CT12_06/SYTOX-G/ANTsR50/ already exists
cfos_CT16_01
/home/gpu_data/data8/cfos_app/2nd/CT16_01/SYTOX-G/ANTsR50/ already exists
cfos_CT16_02
/home/gpu_data/data8/cfos_app/2nd/CT16_02/SYTOX-G/ANTsR50/ already exists
cfos_CT16_03
/home/gpu_data/data8/cfos_app/2nd/CT16_03/SYTOX-G/ANTsR50/ already exists
cfos_CT16_04
/home/gpu_data/data8/cfos_app/2nd/CT16_04/SYTOX-G/ANTsR50/ already exists
cfos_CT16_05
/home/gpu_data/data8/cfos_app/2nd/CT16_05/SYTOX-G/ANTsR50/ already exists
cfos_CT16_06
/home/gpu_data/data8/cfos_app/2nd/CT16_06/SYTOX-G/ANTsR50/ already exists
cfos_CT20_01
/home/gpu_data/data8/cfos_app/2nd/CT20_01/SYTOX-G/ANTsR50/ already exists
cfos_CT20_02
/home/gpu_data/data8/cfos_app/

cfos_CT28_02
/home/gpu_data/data8/cfos_app/1st/CT28_02/SYTOX-G/ANTsR50/ already exists
cfos_CT28_03
/home/gpu_data/data8/cfos_app/1st/CT28_03/SYTOX-G/ANTsR50/ already exists
cfos_CT28_04
/home/gpu_data/data8/cfos_app/1st/CT28_04/SYTOX-G/ANTsR50/ already exists
cfos_CT28_05
/home/gpu_data/data8/cfos_app/1st/CT28_05/SYTOX-G/ANTsR50/ already exists
cfos_CT28_06
/home/gpu_data/data8/cfos_app/1st/CT28_06/SYTOX-G/ANTsR50/ already exists
cfos_CT32_01
/home/gpu_data/data8/cfos_app/1st/CT32_01/SYTOX-G/ANTsR50/ already exists
cfos_CT32_02
/home/gpu_data/data8/cfos_app/1st/CT32_02/SYTOX-G/ANTsR50/ already exists
cfos_CT32_03
/home/gpu_data/data8/cfos_app/1st/CT32_03/SYTOX-G/ANTsR50/ already exists
cfos_CT32_04
/home/gpu_data/data8/cfos_app/1st/CT32_04/SYTOX-G/ANTsR50/ already exists
cfos_CT32_05
/home/gpu_data/data8/cfos_app/1st/CT32_05/SYTOX-G/ANTsR50/ already exists
cfos_CT32_06
/home/gpu_data/data8/cfos_app/1st/CT32_06/SYTOX-G/ANTsR50/ already exists
cfos_CT36_01
/home/gpu_data/data8/cfos_app/

cfos_CT44_01
/home/gpu_data/data8/cfos_app/2nd/CT44_01/SYTOX-G/ANTsR50/ already exists
cfos_CT44_02
/home/gpu_data/data8/cfos_app/2nd/CT44_02/SYTOX-G/ANTsR50/ already exists
cfos_CT44_03
/home/gpu_data/data8/cfos_app/2nd/CT44_03/SYTOX-G/ANTsR50/ already exists
cfos_CT44_04
/home/gpu_data/data8/cfos_app/2nd/CT44_04/SYTOX-G/ANTsR50/ already exists
cfos_CT44_05
/home/gpu_data/data8/cfos_app/2nd/CT44_05/SYTOX-G/ANTsR50/ already exists
cfos_CT44_06
/home/gpu_data/data8/cfos_app/2nd/CT44_06/SYTOX-G/ANTsR50/ already exists
cfos_CT0_01
/home/gpu_data/data8/cfos_app/1st/CT0_01/SYTOX-G/ANTsR50/ already exists
cfos_CT0_02
/home/gpu_data/data8/cfos_app/1st/CT0_02/SYTOX-G/ANTsR50/ already exists
cfos_CT0_03
/home/gpu_data/data8/cfos_app/1st/CT0_03/SYTOX-G/ANTsR50/ already exists
cfos_CT0_04
/home/gpu_data/data8/cfos_app/1st/CT0_04/SYTOX-G/ANTsR50/ already exists
cfos_CT0_05
/home/gpu_data/data8/cfos_app/1st/CT0_05/SYTOX-G/ANTsR50/ already exists
cfos_CT0_06
/home/gpu_data/data8/cfos_app/1st/CT0_06/

cfos_CT8_06
/home/gpu_data/data8/cfos_app/2nd/CT8_06/SYTOX-G/ANTsR50/ already exists
cfos_CT12_01
/home/gpu_data/data8/cfos_app/2nd/CT12_01/SYTOX-G/ANTsR50/ already exists
cfos_CT12_02
/home/gpu_data/data8/cfos_app/2nd/CT12_02/SYTOX-G/ANTsR50/ already exists
cfos_CT12_03
/home/gpu_data/data8/cfos_app/2nd/CT12_03/SYTOX-G/ANTsR50/ already exists
cfos_CT12_04
/home/gpu_data/data8/cfos_app/2nd/CT12_04/SYTOX-G/ANTsR50/ already exists
cfos_CT12_05
/home/gpu_data/data8/cfos_app/2nd/CT12_05/SYTOX-G/ANTsR50/ already exists
cfos_CT12_06
/home/gpu_data/data8/cfos_app/2nd/CT12_06/SYTOX-G/ANTsR50/ already exists
cfos_CT16_01
/home/gpu_data/data8/cfos_app/2nd/CT16_01/SYTOX-G/ANTsR50/ already exists
cfos_CT16_02
/home/gpu_data/data8/cfos_app/2nd/CT16_02/SYTOX-G/ANTsR50/ already exists
cfos_CT16_03
/home/gpu_data/data8/cfos_app/2nd/CT16_03/SYTOX-G/ANTsR50/ already exists
cfos_CT16_04
/home/gpu_data/data8/cfos_app/2nd/CT16_04/SYTOX-G/ANTsR50/ already exists
cfos_CT16_05
/home/gpu_data/data8/cfos_app/2n

cfos_CT24_05
/home/gpu_data/data8/cfos_app/1st/CT24_05/SYTOX-G/ANTsR50/ already exists
cfos_CT24_06
/home/gpu_data/data8/cfos_app/1st/CT24_06/SYTOX-G/ANTsR50/ already exists
cfos_CT28_01
/home/gpu_data/data8/cfos_app/1st/CT28_01/SYTOX-G/ANTsR50/ already exists
cfos_CT28_02
/home/gpu_data/data8/cfos_app/1st/CT28_02/SYTOX-G/ANTsR50/ already exists
cfos_CT28_03
/home/gpu_data/data8/cfos_app/1st/CT28_03/SYTOX-G/ANTsR50/ already exists
cfos_CT28_04
/home/gpu_data/data8/cfos_app/1st/CT28_04/SYTOX-G/ANTsR50/ already exists
cfos_CT28_05
/home/gpu_data/data8/cfos_app/1st/CT28_05/SYTOX-G/ANTsR50/ already exists
cfos_CT28_06
/home/gpu_data/data8/cfos_app/1st/CT28_06/SYTOX-G/ANTsR50/ already exists
cfos_CT32_01
/home/gpu_data/data8/cfos_app/1st/CT32_01/SYTOX-G/ANTsR50/ already exists
cfos_CT32_02
/home/gpu_data/data8/cfos_app/1st/CT32_02/SYTOX-G/ANTsR50/ already exists
cfos_CT32_03
/home/gpu_data/data8/cfos_app/1st/CT32_03/SYTOX-G/ANTsR50/ already exists
cfos_CT32_04
/home/gpu_data/data8/cfos_app/

cfos_CT40_04
/home/gpu_data/data8/cfos_app/2nd/CT40_04/SYTOX-G/ANTsR50/ already exists
cfos_CT40_05
/home/gpu_data/data8/cfos_app/2nd/CT40_05/SYTOX-G/ANTsR50/ already exists
cfos_CT40_06
/home/gpu_data/data8/cfos_app/2nd/CT40_06/SYTOX-G/ANTsR50/ already exists
cfos_CT44_01
/home/gpu_data/data8/cfos_app/2nd/CT44_01/SYTOX-G/ANTsR50/ already exists
cfos_CT44_02
/home/gpu_data/data8/cfos_app/2nd/CT44_02/SYTOX-G/ANTsR50/ already exists
cfos_CT44_03
/home/gpu_data/data8/cfos_app/2nd/CT44_03/SYTOX-G/ANTsR50/ already exists
cfos_CT44_04
/home/gpu_data/data8/cfos_app/2nd/CT44_04/SYTOX-G/ANTsR50/ already exists
cfos_CT44_05
/home/gpu_data/data8/cfos_app/2nd/CT44_05/SYTOX-G/ANTsR50/ already exists
cfos_CT44_06
/home/gpu_data/data8/cfos_app/2nd/CT44_06/SYTOX-G/ANTsR50/ already exists
cfos_CT0_01
/home/gpu_data/data8/cfos_app/1st/CT0_01/SYTOX-G/ANTsR50/ already exists
cfos_CT0_02
/home/gpu_data/data8/cfos_app/1st/CT0_02/SYTOX-G/ANTsR50/ already exists
cfos_CT0_03
/home/gpu_data/data8/cfos_app/1st/C

cfos_CT8_03
/home/gpu_data/data8/cfos_app/2nd/CT8_03/SYTOX-G/ANTsR50/ already exists
cfos_CT8_04
/home/gpu_data/data8/cfos_app/2nd/CT8_04/SYTOX-G/ANTsR50/ already exists
cfos_CT8_05
/home/gpu_data/data8/cfos_app/2nd/CT8_05/SYTOX-G/ANTsR50/ already exists
cfos_CT8_06
/home/gpu_data/data8/cfos_app/2nd/CT8_06/SYTOX-G/ANTsR50/ already exists
cfos_CT12_01
/home/gpu_data/data8/cfos_app/2nd/CT12_01/SYTOX-G/ANTsR50/ already exists
cfos_CT12_02
/home/gpu_data/data8/cfos_app/2nd/CT12_02/SYTOX-G/ANTsR50/ already exists
cfos_CT12_03
/home/gpu_data/data8/cfos_app/2nd/CT12_03/SYTOX-G/ANTsR50/ already exists
cfos_CT12_04
/home/gpu_data/data8/cfos_app/2nd/CT12_04/SYTOX-G/ANTsR50/ already exists
cfos_CT12_05
/home/gpu_data/data8/cfos_app/2nd/CT12_05/SYTOX-G/ANTsR50/ already exists
cfos_CT12_06
/home/gpu_data/data8/cfos_app/2nd/CT12_06/SYTOX-G/ANTsR50/ already exists
cfos_CT16_01
/home/gpu_data/data8/cfos_app/2nd/CT16_01/SYTOX-G/ANTsR50/ already exists
cfos_CT16_02
/home/gpu_data/data8/cfos_app/2nd/CT16

In [ ]:
# Build region lists and colors in graph order

uni_IDs, rev_IDs = ca.get_uni_rIDs()
print(len(uni_IDs))

ex_file = "ex_summary.csv"
df_ex = pd.read_csv(os.path.join(rdir, ex_file), index_col=0)

# Parse RGB triplet string (e.g., "[255, 120, 0]") into 0–1 float array
df_ex["rgb_triplet2"] = df_ex["rgb_triplet"].apply(
    lambda x: np.array(list(map(int, x.strip("[]").split(", ")))) / 255
)

# Graph order IDs
graph_order_ID = df_ex["id"]
uni_IDs_all = [i for i in graph_order_ID if i in uni_IDs]
print("unique all regions", len(uni_IDs_all))

region_colors = [df_ex.loc[df_ex["id"] == rID, "rgb_triplet2"].iloc[0] for rID in uni_IDs_all]
volume_order = np.array([df_ex.loc[df_ex["id"] == rID, "volume"].iloc[0] for rID in uni_IDs_all])

df_sum = df_ex[df_ex["id"].isin(uni_IDs_all)].iloc[:, 0:6]
df_sum = df_sum.sort_values(by="id", key=lambda x: x.map({v: i for i, v in enumerate(uni_IDs_all)}))

print(df_sum)

In [ ]:
# Summarize region-wise cell counts/intensities for each experiment

fpr = 0.5
exps = ["1st", "2nd"]
ants_dir_name = "ANTsR50"

op = f"_ai_fpr{fpr}"
moved_points_id_csv = f"coordinates_ai_id_fpr{fpr}.csv"  # from previous step
combine_points_csv = f"cell_table_combine_I_ai_fpr{fpr}.pkl"

for l, exp in enumerate(exps):
    t_cells = np.load(os.path.join(savedir, exp, f"total_cell_nums_fpr{fpr}.npy"))
    t_intenses = np.load(os.path.join(savedir, exp, f"total_cell_intenses_fpr{fpr}.npy"))

    for i, sample in enumerate(sample_names):
        # Find matched file name just for logging (kept as-is)
        for f in moving_points_csv_li:
            if sample in f.replace("cfos_", ""):
                break
        print(f)

        # Load combined cell table (after ANTs registration)
        moved_points_dir = os.path.join(savedir, exp, sample, "SYTOX-G", ants_dir_name)
        df_comb = pd.read_pickle(os.path.join(moved_points_dir, combine_points_csv))
        print(df_comb.iloc[0:3, :])

        r_count = np.zeros(len(uni_IDs_all), dtype="uint64")
        r_intense = np.zeros(len(uni_IDs_all), dtype="float32")

        for k, rID in enumerate(uni_IDs_all):
            if ca.smallID_q(rID):
                index = np.where(df_comb["atlasID"] == rID)[0]
                r_count[k] = len(index)
                r_intense[k] = np.nansum(np.array(df_comb["intensity"])[index])
            else:
                child_IDs, child_regions, middle_IDs, middle_regions = ca.get_child_IDs2(rID)

                index0 = np.where(df_comb["atlasID"] == rID)[0]
                if len(index0) != 0:
                    r_count_sum = len(index0)
                    r_intense_sum = np.nansum(np.array(df_comb["intensity"])[index0])
                else:
                    r_count_sum = 0
                    r_intense_sum = 0

                for ch_ID in child_IDs + middle_IDs:
                    index = np.where(df_comb["atlasID"] == ch_ID)[0]
                    if len(index) != 0:
                        r_count_sum += len(index)
                        r_intense_sum += np.nansum(np.array(df_comb["intensity"])[index])

                r_count[k] = r_count_sum
                r_intense[k] = r_intense_sum

        # Summary stats over all cells (kept for consistency)
        all_count = len(df_comb)
        all_intense = np.sum(df_comb["intensity"])
        mean_intense = np.mean(df_comb["intensity"])

        # Density / ratios
        r_count_dense = ((r_count.reshape(-1, 1) / volume_order)[:, 0]).astype("float32")
        r_intense_dense = ((r_intense.reshape(-1, 1) / volume_order)[:, 0]).astype("float32")
        r_mean_intense = (r_intense / r_count).astype("float32")
        r_mean_intense = np.nan_to_num(r_mean_intense)

        del df_comb
        gc.collect()

        r_intense_ratio = (r_intense / t_intenses[i]).astype("float32")
        r_intense_ratio_mean = (r_intense / mean_intense).astype("float32")
        r_count_ratio = (r_count / t_cells[i]).astype("float32")

        df_summary = df_sum.copy()
        df_summary["count"] = r_count
        df_summary["intensity"] = r_intense
        df_summary["cell_density"] = r_count_dense
        df_summary["intensity_density"] = r_intense_dense
        df_summary["mean_intensity"] = r_mean_intense
        df_summary["count_ratio"] = r_count_ratio
        df_summary["intensity_ratio"] = r_intense_ratio
        df_summary["intensity_ratio_mean"] = r_intense_ratio_mean

        out_dir = os.path.join(savedir, exp, f"region_cell_intense{op}")
        if not os.path.exists(out_dir):
            os.makedirs(out_dir)
        out_csv = os.path.join(out_dir, f"{sample}_region_cell_intensity_summary.csv")
        df_summary.to_csv(out_csv)
        print(out_csv)
        print(df_summary)

# Combine results from 1st and 2nd experiments
t_cells_1 = np.load(os.path.join(savedir, f"1st/total_cell_nums_fpr{fpr}.npy"))
t_cells_2 = np.load(os.path.join(savedir, f"2nd/total_cell_nums_fpr{fpr}.npy"))
t_cell_T = np.append(t_cells_1, t_cells_2)
print(t_cell_T)
np.save(os.path.join(savedir, f"total_cell_nums_fpr{fpr}"), t_cell_T)

In [ ]:
# Summarize per-sample cell counts and count ratios

fpr = 0.5
exps = ["1st", "2nd"]

op = f"_ai_fpr{fpr}"
for exp in exps:
    df_all_c = []
    df_all_c_r = []
    cols = []

    for sample in sample_names:
        path = os.path.join(savedir, exp, f"region_cell_intense{op}", f"{sample}_region_cell_intensity_summary.csv")
        df = pd.read_csv(path)

        # Keep only the columns actually needed downstream
        df_all_c.append(df["count"])
        df_all_c_r.append(df["count_ratio"])
        cols.append(sample)

    # Count matrix
    df_counts = pd.concat(df_all_c, axis=1)
    df_counts.columns = cols
    df_counts.insert(0, "id", uni_IDs_all)
    df_counts.to_csv(os.path.join(savedir, exp, f"region_cell{op}_count_summary_{exp}.csv"), index=False)

    # Count ratio matrix
    df_count_ratio = pd.concat(df_all_c_r, axis=1)
    df_count_ratio.columns = cols
    df_count_ratio.insert(0, "id", uni_IDs_all)
    df_count_ratio.to_csv(os.path.join(savedir, exp, f"region_cell{op}_count_ratio_summary_{exp}.csv"), index=False)